# NB_lab_test_trends

Build `adb_caresync.gold.lab_test_trends` from the STTM workbook.

* Target grain: one row per test type per calendar month
* Driving table: aggregated `adb_caresync.silver.lab_results`
* Optional date enrichment: `adb_caresync.silver.calendar`
* Load type: full rebuild with `CREATE OR REPLACE TABLE`
* Verified schema adjustments from source inspection:
  * the calendar table uses `datekey`, not `date_key`, so this notebook joins on the first day of each month via `CAST(date_format(month_start_date, 'yyyyMMdd') AS INT)`
  * `result_status` contains the expected `Abnormal` and `Critical` values used by the STTM rules


In [0]:
%sql
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends AS
WITH monthly_aggregates AS (
  SELECT
    test_name,
    YEAR(test_date) AS year,
    MONTH(test_date) AS month,
    DATE_TRUNC('MONTH', test_date) AS month_start_date,
    COUNT(*) AS total_tests,
    COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical') AS critical_test_count
  FROM adb_caresync.silver.lab_results
  GROUP BY
    test_name,
    YEAR(test_date),
    MONTH(test_date),
    DATE_TRUNC('MONTH', test_date)
),
enriched AS (
  SELECT
    ma.test_name,
    ma.year,
    ma.month,
    c.month_name,
    c.quarter_name,
    ma.total_tests,
    ma.abnormal_test_count,
    ma.critical_test_count,
    CAST(ROUND(CASE WHEN ma.total_tests = 0 THEN NULL ELSE ma.abnormal_test_count * 100.0 / ma.total_tests END, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(CASE WHEN ma.total_tests = 0 THEN NULL ELSE ma.critical_test_count * 100.0 / ma.total_tests END, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly_aggregates ma
  LEFT JOIN adb_caresync.silver.calendar c
    ON c.datekey = CAST(date_format(ma.month_start_date, 'yyyyMMdd') AS INT)
)
SELECT
  test_name,
  year,
  month,
  month_name,
  quarter_name,
  total_tests,
  abnormal_test_count,
  critical_test_count,
  abnormal_rate_pct,
  critical_rate_pct,
  LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct,
  CAST(
    ROUND(
      abnormal_rate_pct - LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month),
      2
    ) AS DECIMAL(6,2)
  ) AS abnormal_rate_change_pct
FROM enriched;

In [0]:
processed_rows = spark.table("adb_caresync.gold.lab_test_trends").count()
print(f"processed_rows={processed_rows}")
dbutils.notebook.exit(str(processed_rows))